# AML Transaction Monitoring System

Detecting money laundering in 5 million financial transactions using SQL pattern
analysis, machine learning, and a Power BI triage dashboard. Built on IBM's
synthetic AML dataset, this project combines rule-based detection with a supervised
model and produces a risk-scored alert queue for analysts to investigate.

## The Data

I use IBM's Transactions for Anti-Money-Laundering dataset (HI-Small), a synthetic
but realistic set of 5 million transactions where laundering is labelled. Laundering
makes up only about 0.1% of transactions, mirroring how rare and hidden it is in the
real world.

In [3]:
import kagglehub

# Download the IBM AML dataset
path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'ibm-transactions-for-anti-money-laundering-aml' dataset.
Dataset downloaded to: /kaggle/input/ibm-transactions-for-anti-money-laundering-aml


The dataset has many files (different sizes plus labelled pattern files). I list them
to pick the HI-Small transactions file, which is large enough to be interesting but
manageable.

In [4]:
import os

# List everything in the downloaded folder
for f in os.listdir(path):
    size_mb = os.path.getsize(os.path.join(path, f)) / (1024*1024)
    print(f"{f}  —  {size_mb:.1f} MB")

HI-Medium_accounts.csv  —  138.3 MB
LI-Small_Trans.csv  —  620.3 MB
HI-Large_accounts.csv  —  140.8 MB
HI-Large_Trans.csv  —  16262.8 MB
HI-Medium_Trans.csv  —  2891.3 MB
LI-Medium_accounts.csv  —  135.2 MB
HI-Small_Patterns.txt  —  0.3 MB
HI-Medium_Patterns.txt  —  2.2 MB
HI-Small_accounts.csv  —  32.5 MB
LI-Medium_Trans.csv  —  2838.6 MB
HI-Large_Patterns.txt  —  13.2 MB
LI-Medium_Patterns.txt  —  0.4 MB
LI-Large_accounts.csv  —  137.7 MB
LI-Small_accounts.csv  —  45.1 MB
LI-Large_Patterns.txt  —  1.8 MB
LI-Large_Trans.csv  —  15966.9 MB
LI-Small_Patterns.txt  —  0.1 MB
HI-Small_Trans.csv  —  453.6 MB


## Loading the Transactions

I load the HI-Small transactions into pandas to see what an AML transaction looks
like: sender, receiver, amount, currency, payment format, and the laundering label.

In [5]:
import pandas as pd
import os

# Load the HI-Small transactions file
trans_file = os.path.join(path, "HI-Small_Trans.csv")
df = pd.read_csv(trans_file)

print("Rows, Columns:", df.shape)
df.head()

Rows, Columns: (5078345, 11)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## Loading into SQL

I push the data into a SQLite database so I can explore it with real SQL queries, the
way an analyst would investigate transactions in a bank. The test query confirms 5
million transactions with about 5,000 laundering cases.

In [6]:
import sqlite3

# Create a SQLite database in memory (or a file)
conn = sqlite3.connect("aml.db")

# Clean column names for SQL (remove spaces, dots — SQL doesn't like them)
df.columns = [c.strip().replace(" ", "_").replace(".", "_") for c in df.columns]

# Load the dataframe into a SQL table called 'transactions'
df.to_sql("transactions", conn, if_exists="replace", index=False)

print("Loaded into SQLite. Columns now:")
print(df.columns.tolist())

# Quick test query: count total rows and laundering cases
import pandas as pd
test = pd.read_sql("SELECT COUNT(*) AS total, SUM(Is_Laundering) AS laundering FROM transactions", conn)
print(test)

Loaded into SQLite. Columns now:
['Timestamp', 'From_Bank', 'Account', 'To_Bank', 'Account_1', 'Amount_Received', 'Receiving_Currency', 'Amount_Paid', 'Payment_Currency', 'Payment_Format', 'Is_Laundering']
     total  laundering
0  5078345        5177


## Exploring with SQL: Where Does Laundering Hide?

First I look at the highest-activity accounts. The result is revealing: accounts with
hundreds of thousands of transactions are not launderers, they are banks and large
institutions. So raw transaction volume alone is misleading.

In [7]:
import pandas as pd

query = """
SELECT
    Account AS account,
    COUNT(*) AS num_transactions,
    SUM(Amount_Paid) AS total_paid,
    SUM(Is_Laundering) AS laundering_count
FROM transactions
GROUP BY Account
HAVING COUNT(*) > 50
ORDER BY num_transactions DESC
LIMIT 20;
"""

result = pd.read_sql(query, conn)
print(result)

      account  num_transactions    total_paid  laundering_count
0   100428660            168672  5.276229e+10               243
1   1004286A8            103018  2.606938e+10               158
2   100428978             20497  7.532110e+09                29
3   1004286F0             18663  2.465736e+10                21
4   100428780             17264  3.739337e+11                21
5   1004289C0             16794  1.051719e+10                13
6   100428810             16426  5.153616e+09                26
7   1004287C8             14174  1.011072e+11                16
8   100428738             13756  2.592168e+11                23
9   100428A51             13073  2.628596e+05                18
10  1004288A0             12330  3.177135e+09                16
11  100428858             11000  3.321689e+09                14
12  1004288E8              9471  5.826176e+10                13
13  100428A08              8290  5.818734e+09                13
14  100428930              6431  6.32515

Next I compare the amount signature of laundering versus legitimate transactions.
Laundering amounts average much larger, but the range is wild and multi-currency, so
amount alone is not a clean signal.

In [8]:
import pandas as pd

query = """
SELECT
    Is_Laundering,
    COUNT(*) AS num_transactions,
    AVG(Amount_Paid) AS avg_amount,
    MIN(Amount_Paid) AS min_amount,
    MAX(Amount_Paid) AS max_amount
FROM transactions
GROUP BY Is_Laundering;
"""

result = pd.read_sql(query, conn)
print(result)

   Is_Laundering  num_transactions    avg_amount  min_amount    max_amount
0              0           5073168  4.477000e+06    0.000001  1.046302e+12
1              1              5177  3.613531e+07    0.003227  8.485314e+10


The most useful pattern: accounts where laundering is the majority of their activity.
These turn out to be low-volume, single-purpose mule accounts, often with just one
transaction that is 100% laundering. That is the real laundering signature, the
opposite of the high-volume banks.

In [9]:
import pandas as pd

query = """
SELECT
    Account AS account,
    SUM(CASE WHEN Is_Laundering = 1 THEN 1 ELSE 0 END) AS laundering_txns,
    COUNT(*) AS total_txns,
    ROUND(100.0 * SUM(CASE WHEN Is_Laundering = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS laundering_pct
FROM transactions
GROUP BY Account
HAVING laundering_txns > 0
ORDER BY laundering_pct DESC, laundering_txns DESC
LIMIT 20;
"""

result = pd.read_sql(query, conn)
print(result)

      account  laundering_txns  total_txns  laundering_pct
0   811EDA940                4           4           100.0
1   812D1B280                3           3           100.0
2   80016A6A0                1           1           100.0
3   800934020                1           1           100.0
4   800C67560                1           1           100.0
5   8016221C0                1           1           100.0
6   80189D920                1           1           100.0
7   801908010                1           1           100.0
8   8021199E0                1           1           100.0
9   802E7CA30                1           1           100.0
10  80303F3C0                1           1           100.0
11  803298890                1           1           100.0
12  80330A900                1           1           100.0
13  80342F060                1           1           100.0
14  803B5DA70                1           1           100.0
15  803C83D60                1           1           100

## Saving the SQL Queries

I save these detection queries as a separate .sql file, a portfolio artifact that
visibly demonstrates the SQL work behind the analysis.

In [10]:
sql_queries = """-- AML Transaction Monitoring: SQL Detection Queries
-- Dataset: IBM HI-Small AML (5.07M transactions, 5177 laundering)

-- 1. High-velocity accounts (reveals banks/institutions, not launderers)
SELECT Account, COUNT(*) AS num_transactions,
       SUM(Amount_Paid) AS total_paid,
       SUM(Is_Laundering) AS laundering_count
FROM transactions
GROUP BY Account
HAVING COUNT(*) > 50
ORDER BY num_transactions DESC;

-- 2. Laundering vs legitimate amount signature
SELECT Is_Laundering, COUNT(*) AS num_transactions,
       AVG(Amount_Paid) AS avg_amount,
       MIN(Amount_Paid) AS min_amount,
       MAX(Amount_Paid) AS max_amount
FROM transactions
GROUP BY Is_Laundering;

-- 3. Accounts concentrated in laundering (true mule accounts)
SELECT Account,
       SUM(CASE WHEN Is_Laundering = 1 THEN 1 ELSE 0 END) AS laundering_txns,
       COUNT(*) AS total_txns,
       ROUND(100.0 * SUM(CASE WHEN Is_Laundering = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS laundering_pct
FROM transactions
GROUP BY Account
HAVING laundering_txns > 0
ORDER BY laundering_pct DESC, laundering_txns DESC;
"""

with open("aml_queries.sql", "w") as f:
    f.write(sql_queries)

print("Saved aml_queries.sql — SQL portfolio artifact")

Saved aml_queries.sql — SQL portfolio artifact


## Feature Engineering

I build features from what the SQL exploration taught me. Some are domain guesses
(cross-currency, amount mismatch), others come from the mule-account insight
(same-bank, amount size). The comparison below shows which actually separate
laundering from legitimate activity, and interestingly my cross-currency and amount-
mismatch guesses turn out to be inversely related to laundering, because laundering
here is structurally clean per transaction.

In [11]:
import pandas as pd
import numpy as np

# Work on a manageable sample for modeling (5M rows is heavy on free Colab)
# Keep ALL laundering cases + a sample of legit ones
laundering = df[df['Is_Laundering'] == 1]
legit = df[df['Is_Laundering'] == 0].sample(n=200000, random_state=42)
model_df = pd.concat([laundering, legit]).reset_index(drop=True)

print("Model dataset:", model_df.shape)
print("Laundering cases:", model_df['Is_Laundering'].sum())

# Feature 1: cross-currency flag (paid and received in different currencies = layering signal)
model_df['cross_currency'] = (model_df['Receiving_Currency'] != model_df['Payment_Currency']).astype(int)

# Feature 2: amount mismatch (paid vs received differ — possible obfuscation)
model_df['amount_mismatch'] = (abs(model_df['Amount_Paid'] - model_df['Amount_Received']) > 0.01).astype(int)

# Feature 3: same-bank flag (internal vs external transfer)
model_df['same_bank'] = (model_df['From_Bank'] == model_df['To_Bank']).astype(int)

# Feature 4: log of amount (tames the huge multi-currency range)
model_df['log_amount'] = np.log1p(model_df['Amount_Paid'])

print("\nFeatures created. Comparing laundering vs legit:")
print(model_df.groupby('Is_Laundering')[['cross_currency','amount_mismatch','same_bank','log_amount']].mean())

Model dataset: (205177, 11)
Laundering cases: 5177

Features created. Comparing laundering vs legit:
               cross_currency  amount_mismatch  same_bank  log_amount
Is_Laundering                                                        
0                    0.014085         0.014035   0.136915    7.409999
1                    0.000000         0.000000   0.019896    9.081899


## Adding Account-Level Features

The strongest signal is account behaviour. Using SQL, I compute each account's
transaction count and average amount, then merge them in. Mule accounts have very few
transactions while banks have many, so a low account-activity count points toward
laundering, confirming what the earlier SQL exploration found.

In [12]:
import pandas as pd
import numpy as np

# Compute per-account stats from the FULL dataset (via SQL — efficient)
acct_stats = pd.read_sql("""
    SELECT Account,
           COUNT(*) AS acct_txn_count,
           AVG(Amount_Paid) AS acct_avg_amount
    FROM transactions
    GROUP BY Account
""", conn)

# Merge these account features into our model dataset (on the sender account)
model_df = model_df.merge(acct_stats, on='Account', how='left')

# Feature: log of account's transaction count (mules have very few; banks have many)
model_df['log_acct_txns'] = np.log1p(model_df['acct_txn_count'])

print("Account features merged.")
print(model_df.groupby('Is_Laundering')[['log_acct_txns','log_amount','same_bank']].mean())

Account features merged.
               log_acct_txns  log_amount  same_bank
Is_Laundering                                      
0                   4.058578    7.409999   0.136915
1                   3.531606    9.081899   0.019896


## Supervised Detection (Random Forest)

I train a Random Forest with stratified sampling and balanced class weights to handle
the rare laundering class, then test it on data it has never seen.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Features that carry signal
features = ['log_amount', 'same_bank', 'log_acct_txns', 'acct_avg_amount', 'cross_currency']
X = model_df[features].fillna(0)
y = model_df['Is_Laundering']

# Train/test split, stratified (keeps laundering ratio in both halves)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Train with balanced class weights (rare laundering taken seriously)
rf = RandomForestClassifier(
    n_estimators=50, max_depth=12, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(f"Precision: {precision_score(y_test, y_pred):.1%}")
print(f"Recall:    {recall_score(y_test, y_pred):.1%}")
print(f"F1 score:  {f1_score(y_test, y_pred):.1%}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Precision: 12.4%
Recall:    68.9%
F1 score:  21.0%

Confusion matrix:
[[52452  7549]
 [  483  1070]]


The model catches about 69% of laundering at 12% precision. This is an honest AML
result, not a weakness. Laundering is deliberately designed to look normal, so no
model reaches the 99% of a fraud detector. In practice, compliance teams accept lower
precision to avoid missing laundering, then triage alerts manually, which is exactly
why the dashboard matters.

## Unsupervised Detection (Isolation Forest)

For comparison, I test anomaly detection, which finds statistical outliers without
using labels.

In [14]:
from sklearn.ensemble import IsolationForest

# Train Isolation Forest on the same features (unsupervised — finds outliers)
iso = IsolationForest(contamination=0.02, random_state=42, n_jobs=-1)
model_df['anomaly_score'] = iso.fit_predict(model_df[features].fillna(0))
model_df['is_anomaly'] = (model_df['anomaly_score'] == -1).astype(int)

# How well do anomalies overlap with real laundering?
from sklearn.metrics import precision_score, recall_score
print("Anomaly detection vs laundering:")
print(f"Recall:    {recall_score(model_df['Is_Laundering'], model_df['is_anomaly']):.1%}")
print(f"Precision: {precision_score(model_df['Is_Laundering'], model_df['is_anomaly']):.1%}")

Anomaly detection vs laundering:
Recall:    1.4%
Precision: 1.8%


Anomaly detection performs poorly, catching under 2% of laundering. This confirms
laundering is structured behaviour, not statistical outliers, the same lesson my
fraud-detection project showed. Financial crime needs supervised, pattern-learning
models.

## Risk Scoring and Export

Instead of a flat flag, I score each transaction's laundering probability and tier it
High, Medium, or Low. The High tier holds about 30% precision versus 12% overall and
captures 57% of all laundering, so analysts working the priority queue catch most
laundering while reviewing a fraction of transactions. I export this for the Power BI
dashboard.

In [15]:
import pandas as pd

# Get the model's probability of laundering for every transaction (0 to 1)
model_df['risk_score'] = rf.predict_proba(model_df[features].fillna(0))[:, 1]

# Risk tier for analyst triage
def risk_tier(score):
    if score >= 0.7: return "High"
    elif score >= 0.4: return "Medium"
    else: return "Low"

model_df['risk_tier'] = model_df['risk_score'].apply(risk_tier)

# Build a clean export for Power BI: the suspicious transactions to triage
export = model_df[[
    'Timestamp', 'From_Bank', 'Account', 'To_Bank', 'Account_1',
    'Amount_Paid', 'Payment_Currency', 'Payment_Format',
    'risk_score', 'risk_tier', 'Is_Laundering'
]].copy()

# Sort by risk so the dashboard leads with the most suspicious
export = export.sort_values('risk_score', ascending=False)

# Save to CSV for Power BI
export.to_csv("aml_alerts.csv", index=False)

print("Exported aml_alerts.csv for Power BI")
print("\nRisk tier breakdown:")
print(export['risk_tier'].value_counts())
print("\nLaundering caught in High tier:")
high = export[export['risk_tier'] == 'High']
print(f"{high['Is_Laundering'].sum()} laundering cases in {len(high)} High-risk alerts")

Exported aml_alerts.csv for Power BI

Risk tier breakdown:
risk_tier
Low       162084
Medium     33325
High        9768
Name: count, dtype: int64

Laundering caught in High tier:
2968 laundering cases in 9768 High-risk alerts


I download the alert file to load into Power BI, where I build the analyst triage
dashboard.

In [16]:
from google.colab import files
files.download("aml_alerts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This project combines SQL pattern analysis, supervised machine learning, and a Power
BI triage dashboard into an end-to-end AML monitoring workflow. The key lesson is that
laundering is subtle and structured, so detection depends on learned patterns and
efficient analyst triage rather than any single rule or anomaly score.